# LLM Using Transformer: Complete Workflow with Explanation

This notebook provides a comprehensive understanding of how Large Language Models (LLMs) work using Transformer architecture.

## 1. Introduction to Transformers

Transformers are neural network architectures introduced in the paper "Attention Is All You Need" (2017). They have become the backbone of modern LLMs.

### Key Innovation: Attention Mechanism
- Instead of RNNs processing sequences sequentially, Transformers process entire sequences in parallel
- Uses **Self-Attention** to weigh importance of different tokens
- Allows capturing long-range dependencies efficiently

## 2. Transformer Architecture Overview

```
INPUT TEXT
    ↓
[Tokenization] → Convert text to tokens
    ↓
[Token Embedding] → Convert tokens to vectors
    ↓
[Positional Encoding] → Add position information
    ↓
[Encoder/Decoder Blocks] → Process through attention layers
    ├─ Multi-Head Self-Attention
    ├─ Feed-Forward Network
    └─ Layer Normalization & Residual Connections
    ↓
[Output Layer] → Generate prediction or embedding
    ↓
OUTPUT (Next token, embeddings, etc.)
```

## 3. Self-Attention Mechanism (Core Component)

Self-Attention allows each token to attend to all other tokens in the sequence.

### Mathematical Formula:

```
Attention(Q, K, V) = softmax(QK^T / √d_k) × V

Where:
- Q (Query): What am I looking for?
- K (Key): What information do I have?
- V (Value): What is the actual information?
- d_k: Dimension of key (for scaling)
```

### Step-by-step Process:
1. **Compute Similarity**: QK^T gives attention scores between all pairs of tokens
2. **Scale**: Divide by √d_k to prevent gradient explosion
3. **Normalize**: Apply softmax to get attention weights (0-1, sum to 1)
4. **Apply Weights**: Multiply attention weights with values (V) to get output

## 4. Multi-Head Attention

Instead of single attention, use multiple attention "heads" in parallel.

**Benefits:**
- Different heads learn different relationships
- Head 1 might focus on subject-verb relations
- Head 2 might focus on adjectives
- Combined they capture rich contextual information

```
Input
 ├─ Head 1 → Attention → Output₁
 ├─ Head 2 → Attention → Output₂
 ├─ Head 3 → Attention → Output₃
 └─ Head h → Attention → Outputₕ
           ↓
        Concatenate
           ↓
      Linear Projection
           ↓
       Final Output
```

## 5. Transformer Block (Encoder/Decoder Layer)

Each transformer block contains:

### 5.1 Layer Architecture:

```
INPUT
 ↓
[Multi-Head Self-Attention]
 ↓
[Residual Connection & Layer Norm] → x + Attention(x)
 ↓
[Feed-Forward Network]
    └─ Dense(d_model → d_ff) → ReLU → Dense(d_ff → d_model)
 ↓
[Residual Connection & Layer Norm] → x + FFN(x)
 ↓
OUTPUT
```

### 5.2 Key Components:
- **Residual Connections**: Enable very deep networks
- **Layer Normalization**: Stabilizes training
- **Feed-Forward**: Position-wise dense layers (non-linearity)

## 6. Positional Encoding

Since attention is position-independent, we need to add position information.

### Formula:
```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Where:
- pos: Position in sequence (0, 1, 2, ...)
- i: Dimension index
- d_model: Model dimension
```

### Why this works:
- Different frequencies at different dimensions
- Captures relative and absolute positions
- Helps model understand token order

## 7. LLM Training Workflow

### Step 1: Data Preparation
```
Raw Text → Tokenization → Token IDs → Batches

Example:
"The cat sat on the mat"
↓
[101, 1996, 4251, 2581, 2006, 1996, 12524, 102]
```

### Step 2: Forward Pass
```
Token IDs
   ↓
Token Embedding + Positional Encoding
   ↓
Pass through N Transformer Blocks
   ↓
Output embedding for each token
   ↓
Linear layer projects to vocabulary size
   ↓
Softmax → Probability distribution over tokens
```

### Step 3: Loss Calculation
```
For each position i:
  - Predict next token from tokens 0 to i-1
  - Compare with actual next token (i)
  - Calculate Cross-Entropy Loss
  
Loss = -log(P(actual_token))
```

### Step 4: Backpropagation & Optimization
```
Total Loss
   ↓
Backpropagate through all layers
   ↓
Update weights using optimizer (Adam, SGD, etc.)
   ↓
Repeat for next batch
```

## 8. LLM Inference Workflow (Text Generation)

### Autoregressive Generation:

```
Start with prompt: "The future of AI is"

Step 1: Feed prompt tokens → Get probability of next token
        "The future of AI is" → [0.02, 0.05, 0.15, 0.3, 0.2, ...]
        Select token: "bright" (highest probability)

Step 2: Add new token to sequence
        "The future of AI is bright" → Get next token

Step 3: Repeat until:
        - End token generated [EOS]
        - Max length reached
        - User stops

Final output: "The future of AI is bright and transformative for society"
```

### Decoding Strategies:

**1. Greedy Decoding**
- Select token with highest probability
- Fast but may get stuck in loops

**2. Beam Search**
- Keep top-k most likely sequences
- More diverse and better quality

**3. Temperature Sampling**
- High temperature: More random, creative
- Low temperature: More deterministic, coherent

**4. Top-k / Top-p Sampling**
- Only sample from top-k most likely tokens
- Only sample from tokens with cumulative prob > p
- Better balance between quality and diversity

## 9. Complete LLM Pipeline Summary

### Training Pipeline:
```
Raw Text Corpus
    ↓
Tokenization (BPE/WordPiece/SentencePiece)
    ↓
Create Dataset (prepare input-output pairs)
    ↓
Mini-batches
    ├─ Embedding Layer: Token IDs → Dense Vectors
    ├─ Add Positional Encoding
    ├─ Pass through Transformer Encoder Blocks (×N)
    │  Each block:
    │  - Multi-Head Self-Attention
    │  - Feed-Forward Network
    │  - Residual + Layer Norm
    ├─ Output Projection (to vocab size)
    └─ Softmax → Token Probabilities
    ↓
Calculate Loss (Cross-Entropy)
    ↓
Backpropagation
    ↓
Weight Update (Adam optimizer)
    ↓
Repeat for millions of batches
    ↓
Trained LLM
```

### Inference Pipeline:
```
User Input (Prompt)
    ↓
Tokenize Input
    ↓
Feed tokens through trained model
    ↓
Get probability distribution for next token
    ↓
Sample/Select next token using decoding strategy
    ↓
Append token to sequence
    ↓
Repeat until [EOS] or max_length
    ↓
Detokenize (convert tokens back to text)
    ↓
Return Generated Text
```

## 10. Key Hyperparameters

| Parameter | Meaning | Typical Value |
|-----------|---------|---------------|
| d_model | Embedding dimension | 768 - 12288 |
| num_heads | Number of attention heads | 8 - 128 |
| num_layers | Number of transformer blocks | 12 - 96 |
| d_ff | Feed-forward hidden dimension | 2048 - 49152 |
| max_seq_length | Maximum input sequence length | 512 - 8192 |
| vocab_size | Number of unique tokens | 10000 - 100000 |
| dropout | Regularization rate | 0.1 - 0.3 |
| learning_rate | Training step size | 1e-5 - 1e-3 |
| batch_size | Samples per training step | 32 - 4096 |

In [ ]:
# Simple Attention Mechanism Implementation
import numpy as np
from scipy.special import softmax

def scaled_dot_product_attention(Q, K, V):
    """
    Compute scaled dot-product attention
    
    Args:
        Q: Query matrix (seq_len, d_k)
        K: Key matrix (seq_len, d_k)
        V: Value matrix (seq_len, d_v)
    
    Returns:
        Attention output, Attention weights
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute similarity scores
    scores = np.matmul(Q, K.T) / np.sqrt(d_k)
    
    # Step 2: Apply softmax to get attention weights
    attention_weights = softmax(scores, axis=-1)
    
    # Step 3: Apply weights to values
    output = np.matmul(attention_weights, V)
    
    return output, attention_weights

# Example
seq_len = 4  # sequence length
d_k = 8      # key dimension
d_v = 8      # value dimension

Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)
print("\nAttention weights (normalized):")
print(weights)

## 11. Advantages of Transformer-based LLMs

✅ **Parallelization**: All tokens processed simultaneously (not sequentially like RNN)

✅ **Long-range dependencies**: Attention can capture relationships over long distances

✅ **Scalability**: Can be scaled to billions of parameters

✅ **Transfer learning**: Pre-trained models can be fine-tuned for downstream tasks

✅ **Interpretability**: Attention weights show what model focuses on

❌ **Limitations**: Quadratic complexity O(n²) for sequence length, Memory requirements

## 12. Real-World Examples

### Popular Transformer-based LLMs:

1. **BERT** (Bidirectional Encoder Representations)
   - Encoder-only, 12-24 layers
   - Pre-trained on masked language modeling
   - Used for classification, NER, Q&A

2. **GPT Series** (Generative Pre-trained Transformer)
   - Decoder-only, 12-176 billion parameters
   - Trained on next-token prediction
   - Used for text generation, few-shot learning

3. **T5** (Text-to-Text Transfer Transformer)
   - Encoder-Decoder, treats all tasks as text-to-text
   - Versatile for translation, summarization, Q&A

4. **LLaMA / Phi / Mistral**
   - Smaller, efficient models
   - Open-source alternatives to GPT

### Typical Use Cases:
- Text generation (chat, creative writing)
- Machine translation
- Question answering
- Sentiment analysis
- Code generation
- Summarization

## 13. Step-by-Step Example: Generating Text

### Input: "Machine learning is"

```
STEP 1: Tokenization
"Machine learning is" → [Machine, learning, is]
                      → [5023, 4673, 2003]  (token IDs)

STEP 2: Embedding + Positional Encoding
Token ID 5023 → 768-dim embedding vector
Add position encoding for position 0
→ [0.2, -0.5, 0.1, ... ] (768 values)

STEP 3: Forward through Transformer (12 layers)
Each layer applies:
  - Multi-head attention (8 heads)
  - Feed-forward (3072 hidden units)
  - Normalization & residuals
Input: [0.2, -0.5, 0.1, ...]
Output: [0.1, -0.3, 0.2, ...] (transformed)

STEP 4: Output for last token "is"
Take output embedding for position 2
Project to vocabulary size (50257 tokens)
→ [logits for each possible token]

STEP 5: Softmax → Probabilities
logits → [0.01, 0.02, 0.05, 0.15, 0.25, ...]
                                  ↑
                            top prediction

STEP 6: Select Next Token
Sample from distribution OR take argmax
Result: token ID 8899 → "fascinating"

STEP 7: Repeat
"Machine learning is fascinating"
Continue until [EOS] token

FINAL OUTPUT:
"Machine learning is fascinating and transformative technology
that impacts every industry globally."
```

## 14. Summary

### The Complete LLM Workflow:

1. **Tokenization**: Text → token IDs
2. **Embedding**: Token IDs → dense vectors
3. **Positional Encoding**: Add position information
4. **Transformer Layers**: Apply self-attention + feed-forward N times
5. **Output Projection**: Hidden state → vocabulary logits
6. **Softmax**: Logits → probability distribution
7. **Sampling/Selection**: Pick next token
8. **Autoregressive Loop**: Repeat from step 2 with new token appended
9. **Detokenization**: Token IDs → readable text

### Key Insight:
**Transformers excel at capturing context and relationships through attention, enabling powerful language models to understand and generate human-like text.**